In [ ]:
from pathlib import Path
import os
import sys

root = next(path for path in [Path.cwd(), *Path.cwd().parents] if (path / 'config/paper.json').is_file())
os.chdir(root)
if str(root) not in sys.path:
    sys.path.insert(0, str(root))

import numpy as np
import pandas as pd
from sklearn.metrics import roc_auc_score, average_precision_score, precision_score, recall_score, f1_score, matthews_corrcoef
from IPython.display import display
n_bootstrap = 5000
block_length = 10
confidence_level = 0.95
random_seed = 20261027
output = Path('Results/Confidence Intervals')

folder = Path('FINDER/Official')
scores = pd.read_csv(folder / 'daily_scores_all_markets.csv')
metrics = pd.read_csv(folder / 'market_agent_metrics.csv')
thresholds = pd.read_csv(folder / 'daily_thresholds.csv')
rows = []
for (market, agent), frame in thresholds.groupby(['market', 'agent']):
    source = scores[scores.market.eq(market)].set_index('target_date').loc[frame.target_date]
    actual = source.actual_label.to_numpy()
    probability = source[agent].to_numpy()
    predicted = (probability >= frame.threshold.to_numpy()).astype(int)
    assert np.array_equal(predicted, frame.prediction)
    row = {'market': market, 'agent': agent, 'roc_auc': roc_auc_score(actual, probability), 'ap': average_precision_score(actual, probability), 'precision': precision_score(actual, predicted, zero_division=0), 'recall': recall_score(actual, predicted, zero_division=0), 'f1': f1_score(actual, predicted, zero_division=0), 'mcc': matthews_corrcoef(actual, predicted)}
    expected = metrics[metrics.market.eq(market) & metrics.agent.eq(agent)].iloc[0]
    for field in ['roc_auc', 'ap', 'precision', 'recall', 'f1', 'mcc']:
        assert np.isclose(row[field], expected[field])
    rows.append(row)
calculated = pd.DataFrame(rows)
display(calculated)


In [ ]:
markets = {'^GSPC': 'SP500', 'FTSE100': 'FTSE', 'GDAXI': 'DAX', 'CACT': 'CAC', 'STOXX': 'STOXX', 'CBU0.L': 'CBU0'}
models = ['decision_supervisor', 'quantitative_expert', 'chart_expert', 'news_reader', 'news_expert', 'lstm_ind', 'lstm_news']
metric_names = ['roc_auc', 'ap']
frames = []
for market, code in markets.items():
    frame = scores.loc[scores.market.eq(market), ['target_date', 'actual_label', *models[:5]]].copy()
    assert len(frame) and not frame.target_date.duplicated().any(), market
    for variant, model in [('IND', 'lstm_ind'), ('NEWS', 'lstm_news')]:
        benchmark = pd.read_csv(Path('Benchmark/Official') / f'{code}_{variant}' / 'test_predictions.csv')
        assert set(frame.target_date) == set(benchmark.target_date), (market, model)
        frame = frame.merge(benchmark[['target_date', 'actual', 'probability']], on='target_date', validate='one_to_one')
        assert frame.actual_label.eq(frame.actual).all(), (market, model)
        frame = frame.drop(columns='actual').rename(columns={'probability': model})
    frame['target_date'] = pd.to_datetime(frame.target_date)
    frame = frame.sort_values('target_date').set_index('target_date')
    assert frame.index.is_unique and frame.actual_label.isin([0, 1]).all(), market
    assert frame.actual_label.nunique() == 2 and np.isfinite(frame.to_numpy(dtype=float)).all(), market
    frames.append(frame)

dates = pd.DatetimeIndex(sorted(set().union(*(frame.index for frame in frames))))
panel = np.stack([frame.reindex(dates)[['actual_label', *models]].to_numpy(dtype=float) for frame in frames])
sample_sizes = np.array([len(frame) for frame in frames])
event_counts = np.array([frame.actual_label.sum() for frame in frames], dtype=int)
point_metrics = np.empty((len(markets) + 1, len(models), len(metric_names)))
benchmark_metrics = pd.read_csv('Benchmark/Official/metrics.csv')
for m, (market, code) in enumerate(markets.items()):
    actual = frames[m].actual_label.to_numpy()
    for j, model in enumerate(models):
        probability = frames[m][model].to_numpy()
        point_metrics[m, j] = [roc_auc_score(actual, probability), average_precision_score(actual, probability)]
        if model.startswith('lstm_'):
            variant = 'IND' if model == 'lstm_ind' else 'NEWS'
            expected = benchmark_metrics.loc[benchmark_metrics.market.eq(code) & benchmark_metrics.model.eq(variant)]
        else:
            expected = metrics.loc[metrics.market.eq(market) & metrics.agent.eq(model)]
        assert len(expected) == 1 and np.allclose(point_metrics[m, j], expected[metric_names].iloc[0]), (market, model)
point_metrics[-1] = point_metrics[:-1].mean(axis=0)
display(pd.DataFrame({'market': list(markets), 'n': sample_sizes, 'events': event_counts, 'event_rate': event_counts / sample_sizes}))


In [ ]:
assert isinstance(n_bootstrap, int) and n_bootstrap >= 2
assert 1 <= block_length <= len(dates) and 0 < confidence_level < 1
rng = np.random.default_rng(random_seed)
bootstrap_metrics = np.full((n_bootstrap, len(markets) + 1, len(models), len(metric_names)), np.nan)

# Stationary bootstrap on the union-date grid; all markets and models share each draw.
# Block length is measured in union-date observations, with circular continuation.
for b in range(n_bootstrap):
    indices = rng.integers(len(dates), size=len(dates))
    restart = rng.random(len(dates)) < 1 / block_length
    for t in range(1, len(dates)):
        if not restart[t]:
            indices[t] = (indices[t - 1] + 1) % len(dates)
    for m in range(len(markets)):
        sample = panel[m, indices]
        sample = sample[np.isfinite(sample[:, 0])]
        actual = sample[:, 0]
        if np.unique(actual).size < 2:
            continue
        for j in range(len(models)):
            probability = sample[:, j + 1]
            bootstrap_metrics[b, m, j] = [roc_auc_score(actual, probability), average_precision_score(actual, probability)]
    # Ordinary mean keeps a draw invalid if any market lacks either label class.
    bootstrap_metrics[b, -1] = bootstrap_metrics[b, :-1].mean(axis=0)
    if (b + 1) % 1000 == 0:
        print(f'Bootstrap: {b + 1}/{n_bootstrap}')

point_differences = point_metrics[:, :1, :] - point_metrics[:, 1:, :]
bootstrap_differences = bootstrap_metrics[:, :, :1, :] - bootstrap_metrics[:, :, 1:, :]


In [ ]:
metric_rows, difference_rows = [], []
quantiles = [(1 - confidence_level) / 2, (1 + confidence_level) / 2]
settings = {'method': 'stationary_bootstrap_percentile', 'confidence_level': confidence_level, 'n_bootstrap': n_bootstrap, 'block_length': block_length, 'block_unit': 'union_date_observations', 'random_seed': random_seed}

for m, market in enumerate([*markets, 'equal_weight_mean']):
    aggregate = m == len(markets)
    context = {'market': market, 'n': None if aggregate else int(sample_sizes[m]), 'events': None if aggregate else int(event_counts[m])}
    context['event_rate'] = float((event_counts / sample_sizes).mean() if aggregate else event_counts[m] / sample_sizes[m])
    context['aggregation'] = 'equal_weight_6_markets' if aggregate else 'within_market'
    for j, model in enumerate(models):
        for k, metric in enumerate(metric_names):
            values = bootstrap_metrics[:, m, j, k]
            valid = values[np.isfinite(values)]
            bounds = np.quantile(valid, quantiles) if len(valid) >= 2 else [np.nan, np.nan]
            metric_rows.append({**context, 'model': model, 'metric': metric, 'estimate': point_metrics[m, j, k], 'ci_lower': bounds[0], 'ci_upper': bounds[1], 'n_valid': len(valid), 'n_invalid': n_bootstrap - len(valid), **settings})
    for j, comparator in enumerate(models[1:]):
        for k, metric in enumerate(metric_names):
            values = bootstrap_differences[:, m, j, k]
            valid = values[np.isfinite(values)]
            bounds = np.quantile(valid, quantiles) if len(valid) >= 2 else [np.nan, np.nan]
            difference_rows.append({**context, 'model': models[0], 'comparator': comparator, 'metric': metric, 'estimate': point_differences[m, j, k], 'ci_lower': bounds[0], 'ci_upper': bounds[1], 'n_valid': len(valid), 'n_invalid': n_bootstrap - len(valid), **settings})

metrics_ci = pd.DataFrame(metric_rows)
paired_differences_ci = pd.DataFrame(difference_rows)
output.mkdir(parents=True, exist_ok=True)
metrics_ci.to_csv(output / 'metrics_ci.csv', index=False)
paired_differences_ci.to_csv(output / 'paired_differences_ci.csv', index=False)

# Pointwise intervals conditional on saved predictions; no training or LLM variability.
# Aggregate event_rate is the mean of market prevalences; n/events apply per market only.
print(f'{confidence_level:.0%} pointwise intervals; paired differences = Supervisor minus comparator.')
print(f'Saved {len(metrics_ci)} metric intervals and {len(paired_differences_ci)} paired intervals to {output}.')
display(metrics_ci.loc[metrics_ci.model.eq(models[0]), ['market', 'metric', 'estimate', 'ci_lower', 'ci_upper', 'n_valid', 'n_invalid']].round(4))
display(paired_differences_ci.loc[paired_differences_ci.market.eq('equal_weight_mean'), ['comparator', 'metric', 'estimate', 'ci_lower', 'ci_upper', 'n_valid', 'n_invalid']].round(4))


In [ ]:
evaluate_new_run = False
run_id = 'concept'
ticker = '^GSPC'

if evaluate_new_run:
    from finder_core.settings import load_settings
    from finder_core.public_outputs import evaluate
    settings = load_settings(root, ticker=ticker)
    scores, metrics = evaluate(root / 'Runs' / ticker / run_id, root / settings['paths']['labels'])
    display(metrics)
